In [2]:
import pandas as pd
import numpy as np
from MyTfIdfVectorizer import MyTfIdfVectorizer 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import re


# Part E: Kiểm thử  trên corpus nhỏ và so sánh với TfidfVectorizer của sklearn

In [3]:
data = pd.DataFrame({
    "text": [
        "cat eats fish",
        "dog eats fish",
        "cat likes fish"
    ]
})

my_vectorizer = MyTfIdfVectorizer(data)
sklearn_vectorizer = TfidfVectorizer()

## 0.1 My Tf-Idf

In [4]:
my_vectorizer.buildVocabulary("text")
expected_vocab = [
    "cat",
    "dog",
    "eats",
    "fish",
    "likes"
]

assert my_vectorizer.vocab == expected_vocab


In [5]:
my_vectorizer.compute_counts("text")

expected_counts = np.array([
    [1, 0, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [1, 0, 0, 1, 1]
], dtype=np.float32)

assert np.array_equal(
    my_vectorizer.count_matrix.toarray(),
    expected_counts
)

In [6]:
# compute_tf
my_vectorizer.compute_tf()

expected_tf = np.array([
    [1/3, 0,   1/3, 1/3, 0],
    [0,   1/3, 1/3, 1/3, 0],
    [1/3, 0,   0,   1/3, 1/3]
], dtype=np.float32)

assert np.allclose(
    my_vectorizer.tf.toarray(),
    expected_tf,
    atol=1e-9
)

In [7]:
# compute_df
my_vectorizer.compute_df()

expected_df = np.array([2, 1, 2, 3, 1])

assert np.array_equal(
    my_vectorizer.df,
    expected_df
)


In [8]:
# compute idf
my_vectorizer.compute_idf()

expected_idf = np.log(
    4 / (1 + expected_df)
) + 1

assert np.allclose(
    my_vectorizer.idf,
    expected_idf,
    atol=1e-9
)


In [9]:
# compute_tfidf

my_vectorizer.compute_tfidf()

raw_tfidf = expected_tf * expected_idf

norms = np.linalg.norm(
    raw_tfidf,
    axis=1,
    keepdims=True
)

expected_tfidf = raw_tfidf / norms
print(expected_tfidf)
print(my_vectorizer.tfidf)
assert np.allclose(
    my_vectorizer.tfidf.toarray(),
    expected_tfidf,
    atol=1e-9
)

[[0.61980538 0.         0.61980538 0.48133417 0.        ]
 [0.         0.72033345 0.54783215 0.42544054 0.        ]
 [0.54783215 0.         0.         0.42544054 0.72033345]]
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 3)	0.4813341733388614
  (0, 2)	0.6198053781535761
  (0, 0)	0.6198053781535761
  (1, 3)	0.425440540311846
  (1, 2)	0.5478321498361728
  (1, 1)	0.7203334521352189
  (2, 4)	0.7203334521352189
  (2, 3)	0.425440540311846
  (2, 0)	0.5478321498361728


In [10]:
#compute_cosine_similarity
my_vectorizer.compute_cosine_similarity()

# Test diagonal = 1
assert np.allclose(
    np.diag(my_vectorizer.cosine_similarity),
    np.ones(3),
    atol=1e-9
)
print(my_vectorizer.cosine_similarity)
# Test symmetry
assert np.allclose(
    my_vectorizer.cosine_similarity,
    my_vectorizer.cosine_similarity.T,
    atol=1e-9
)


Check running
[[1.         0.54432838 0.54432838]
 [0.54432838 1.         0.18099965]
 [0.54432838 0.18099965 1.        ]]


## 0.2 Compare with sklearn TfIdfVectorizer

In [11]:
sklearn_vectorizer = TfidfVectorizer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
)
cv = CountVectorizer()
count_matrix = cv.fit_transform(data["text"])

row_sums = np.asarray(count_matrix.sum(axis=1)).ravel()

sklearn_tf = count_matrix.multiply(
    1 / row_sums[:, None]
)
sklearn_tfidf = sklearn_vectorizer.fit_transform(data["text"])
sklearn_idf = sklearn_vectorizer.idf_

print((my_vectorizer.tf))
print((sklearn_tf))
print(type(my_vectorizer.idf))
print(type(my_vectorizer.tfidf))


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 3)	0.3333333333333333
  (0, 2)	0.3333333333333333
  (0, 0)	0.3333333333333333
  (1, 3)	0.3333333333333333
  (1, 2)	0.3333333333333333
  (1, 1)	0.3333333333333333
  (2, 4)	0.3333333333333333
  (2, 3)	0.3333333333333333
  (2, 0)	0.3333333333333333
<COOrdinate sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>
  Coords	Values
  (0, 0)	0.3333333333333333
  (0, 2)	0.3333333333333333
  (0, 3)	0.3333333333333333
  (1, 2)	0.3333333333333333
  (1, 3)	0.3333333333333333
  (1, 1)	0.3333333333333333
  (2, 0)	0.3333333333333333
  (2, 3)	0.3333333333333333
  (2, 4)	0.3333333333333333
<class 'numpy.ndarray'>
<class 'scipy.sparse._csr.csr_matrix'>


In [12]:
def compare_matrices(name, sklearn_result, my_result, atol=1e-6):

    if hasattr(sklearn_result, "toarray"):
        sklearn_result = sklearn_result.toarray()

    print(f"{name}:")
    print("  Shape:", sklearn_result.shape, my_result.shape)

    print(
        "  Equal:",
        np.allclose(
            sklearn_result,
            my_result,
            atol=atol
        )
    )

    print(
        "  Max error:",
        np.max(
            np.abs(
                sklearn_result - my_result
            )
        )
    )

In [13]:
compare_matrices(
    "TF",
    sklearn_tf,
    my_vectorizer.tf.toarray()
)

compare_matrices(
    "IDF",
    sklearn_idf,
    my_vectorizer.idf
)

compare_matrices(
    "TF-IDF",
    sklearn_tfidf,
    my_vectorizer.tfidf.toarray()
)


TF:
  Shape: (3, 5) (3, 5)
  Equal: True
  Max error: 0.0
IDF:
  Shape: (5,) (5,)
  Equal: True
  Max error: 1.602477883722031e-08
TF-IDF:
  Shape: (3, 5) (3, 5)
  Equal: True
  Max error: 5.0912635218836044e-09


# Part F: Preprocessing Ablation

In [14]:
df = pd.read_json("/home/vitquay1708/Study_Space/NLP/lab1/c4-train.00000-of-01024-30K.json.gz", lines=True)
df.head(5)

,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25 12:57:54+00:00,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21 10:07:13+00:00,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25 10:40:23+00:00,https://awishcometrue.com/Catalogs/Clearance/T...
3,How many backlinks per day for new site?\nDisc...,2019-04-21 12:46:19+00:00,https://www.blackhatworld.com/seo/how-many-bac...
4,The Denver Board of Education opened the 2017-...,2019-04-20 14:33:21+00:00,http://bond.dpsk12.org/category/news/


## 1. Minimal

In [15]:
df['text_lower'] = df['text'].str.lower()
df['pipeline_1'] = df['text_lower'].apply(lambda x: x.split())
display(df[['text', 'text_lower', 'pipeline_1']].head(3))


,text,text_lower,pipeline_1
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[beginners, bbq, class, taking, place, in, mis..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, 'mac, os, x, lion, (10.7)', s..."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, lycra, and, spandex, shortall, w..."


## 2. Pipeline B - Normalized

In [16]:
df['pipeline_2'] = df['text_lower'].apply(lambda x: re.findall(r'\b\w+\b', x))
display(df[['text', 'text_lower', 'pipeline_2']].head(3))

,text,text_lower,pipeline_2
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[beginners, bbq, class, taking, place, in, mis..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, mac, os, x, lion, 10, 7, star..."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, lycra, and, spandex, shortall, w..."


## 3 Pipeline C - Extended

In [17]:
from transformers import AutoTokenizer

subword_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

df['pipeline_3'] = df['text_lower'].apply(lambda x: subword_tokenizer.tokenize(x))
display(df[['text', 'text_lower', 'pipeline_3']].head(3))

/home/vitquay1708/miniconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


,text,text_lower,pipeline_3
0,Beginners BBQ Class Taking Place in Missoula!\...,beginners bbq class taking place in missoula!\...,"[begin, ##ners, bb, ##q, class, taking, place,..."
1,Discussion in 'Mac OS X Lion (10.7)' started b...,discussion in 'mac os x lion (10.7)' started b...,"[discussion, in, ', mac, os, x, lion, (, 10, ...."
2,Foil plaid lycra and spandex shortall with met...,foil plaid lycra and spandex shortall with met...,"[foil, plaid, l, ##y, ##cr, ##a, and, span, ##..."


## Result Comparison

In [18]:
from collections import Counter

def tokenizer1(text):
    return text.split()

def tokenizer2(text):
    return re.findall(r'\b\w+\b', text)

def tokenizer3(text):
    return subword_tokenizer.tokenize(text)

tokenizers = {'pipeline_1':tokenizer1, 'pipeline_2':tokenizer2, 'pipeline_3':tokenizer3}
def calculate_oov_rate(tokenized_docs, vocab):
    total = 0
    oov = 0
    for doc in tokenized_docs:
        for token in doc:
            total += 1
            if token not in vocab:
                oov+=1
    return oov/total if total > 0 else 0

def compare_pipelines(
    df,
    pipeline_columns,
    test_docs,
    tokenizers
):
    results = {}

    for _, column in enumerate(pipeline_columns):

        docs = df[column].dropna().tolist()
        vocabulary = set(
            token
            for doc in docs
            for token in doc
        )
        vocab_size = len(vocabulary)
        avg_tokens = (
            sum(len(doc) for doc in docs) / len(docs)
            if docs else 0
        )
        n_docs = len(docs)

        non_zero = sum(
            len(set(doc))
            for doc in docs
        )

        total_entries = n_docs * vocab_size

        sparsity = (
            1 - non_zero / total_entries
            if total_entries > 0
            else 0
        )

        test_tokenized = [
            tokenizers[column](text)
            for text in test_docs
        ]

        oov_rate = calculate_oov_rate(test_tokenized, vocabulary)

        results[column] = {
            "Vocabulary size": vocab_size,
            "Average tokens/document": avg_tokens,
            "Matrix sparsity": sparsity,
            "OOV rate": oov_rate
        }

    return pd.DataFrame(results).T

In [19]:
test_docs = [
    "The cat is running quickly.",
    "I love machine learning.",
    "This is an amazing example.",
    "Natural language processing is interesting.",
    "The model generates new tokens.",
    "Students are learning artificial intelligence.",
    "The weather is beautiful today.",
    "This sentence contains an unknownword.",
    "Transformers are widely used in NLP.",
    "Subword tokenization handles rare words."
]
result_comparison = compare_pipelines(df, ['pipeline_1', 'pipeline_2', 'pipeline_3'],test_docs,tokenizers)

display(result_comparison)

,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate
pipeline_1,473388.0,361.089067,0.999611,0.24
pipeline_2,193837.0,369.696367,0.999122,0.24
pipeline_3,28339.0,465.099200,0.993200,0.00


# Part G: Document Search Engine

## 1. Building TF-IDF index on 30K documents

In [20]:
import time
start_time = time.time()

search_engine = MyTfIdfVectorizer(df)
search_engine.fit("text")
elapsed = time.time() - start_time
print(f"Done in {elapsed:.2f}s")
print(f"Documents: {search_engine.tfidf.shape[0]}")
print(f"Vocabulary size: {len(search_engine.vocab)}")

Done in 19.28s
Documents: 30000
Vocabulary size: 254766


## 2. Queries

In [28]:
user_query = input("Enter your search query: ")
results = search_engine.search(user_query, top_k=5)
print(f'\nResults for: "{user_query}"')
display(results)





Results for: "technology"


,Rank,Document ID,Similarity,Document Preview
0,1,23470,0.3653,UTRS has helped provide management and operati...
1,2,11911,0.3366,What exactly is point of sale technology? I li...
2,3,3751,0.3263,"YOKOHAMA, Japan – Nissan Motor Co. announced i..."
3,4,13199,0.3208,"Information Technology, Data Centers, Audio/Vi..."
4,5,26444,0.2954,Practical Action believes that development cha...


## Using another tokenizer

In [22]:
start_time = time.time()
search_engine_bert = MyTfIdfVectorizer(df, subword_tokenizer.tokenize)
search_engine_bert.fit("text")
elapsed = time.time() - start_time
print(f"Done in {elapsed:.2f}s")
print(f"Documents: {search_engine_bert.tfidf.shape[0]}")
print(f"Vocabulary size: {len(search_engine_bert.vocab)}")


Done in 199.12s
Documents: 30000
Vocabulary size: 28339


In [23]:
print(len(search_engine_bert.vocab))

28339


In [29]:
user_query = input("Enter your search query: ")
results = search_engine_bert.search(user_query, top_k=5)
print(f'\nResults for: "{user_query}"')
display(results)


Results for: "health"


,Rank,Document ID,Similarity,Document Preview
0,1,9660,0.4954,"Collection, use, access and/or disclosure of p..."
1,2,14038,0.4919,using the devices for managing their health. F...
2,3,3610,0.4764,Heather is a health economist and health servi...
3,4,27599,0.4167,Health Plan One offers several health insuranc...
4,5,8622,0.4086,Start your journey to better health with Rosem...


# Part H: Evaluation

## 1. Load Evaluation Set

In [ ]:
import ast
eval_df = pd.read_csv('evaluation_set.csv')
eval_df['doc_ids'] = eval_df['doc_ids'].apply(ast.literal_eval)
display(eval_df)

,query_id,query,doc_ids
0,1,education university college students academic...,"[2978, 26379, 25692, 5094, 24368]"
1,2,car vehicle driving automotive auto insurance,"[2640, 3358, 2208, 26928, 13861]"
2,3,real estate property housing apartment bedroom...,"[5366, 6300, 28194, 8703, 22349]"
3,4,health medical doctor patient disease hospital,"[28824, 15913, 13861, 16199, 14038]"
4,5,software computer programming developer algori...,"[20599, 15236, 13825, 11641, 18521]"
5,6,recipe cooking delicious ingredient restaurant...,"[24923, 21688, 2377, 11005, 15225]"
6,7,travel flight hotel tourist destination vacation,"[9722, 2321, 4286, 4835, 18448]"
7,8,sports fitness workout football basketball ath...,"[19879, 22803, 8690, 29529, 20390]"
8,9,fashion clothing wardrobe outfit boutique apparel,"[28168, 6940, 18978, 24634, 10311]"
9,10,investment finance stock market portfolio banking,"[23661, 16358, 2279, 19024, 6300]"


## 2. Define Evaluation Metrics

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k=5):
    """P@K = #relevant documents retrieved / K"""
    retrieved_at_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    relevant_retrieved = sum(1 for doc_id in retrieved_at_k if doc_id in relevant_set)
    return relevant_retrieved / k

def recall_at_k(retrieved_ids, relevant_ids, k=5):
    """R@K = #relevant documents retrieved / #relevant documents"""
    retrieved_at_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)
    relevant_retrieved = sum(1 for doc_id in retrieved_at_k if doc_id in relevant_set)
    return relevant_retrieved / len(relevant_set) if len(relevant_set) > 0 else 0.0

def reciprocal_rank(retrieved_ids, relevant_ids):
    """RR = 1 / rank of first relevant document"""
    relevant_set = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0

## 3. Evaluate Search Engines

In [ ]:
def evaluate_search_engine(engine, eval_df, k=5):
    """Evaluate a search engine on the evaluation set.
    Returns per-query metrics and aggregated metrics."""
    per_query_results = []
    for _, row in eval_df.iterrows():
        query = row['query']
        relevant_ids = row['doc_ids']
        # Run search
        results = engine.search(query, top_k=k)
        retrieved_ids = results['Document ID'].tolist()
        # Compute metrics
        p_at_k = precision_at_k(retrieved_ids, relevant_ids, k)
        r_at_k = recall_at_k(retrieved_ids, relevant_ids, k)
        rr = reciprocal_rank(retrieved_ids, relevant_ids)
        per_query_results.append({
            'query_id': row['query_id'],
            'query': query,
            'retrieved_ids': retrieved_ids,
            'relevant_ids': relevant_ids,
            f'P@{k}': p_at_k,
            f'R@{k}': r_at_k,
            'RR': rr
        })
    results_df = pd.DataFrame(per_query_results)
    # Compute aggregated metrics
    mean_p_at_k = results_df[f'P@{k}'].mean()
    mean_r_at_k = results_df[f'R@{k}'].mean()
    mrr = results_df['RR'].mean()
    return results_df, {
        f'Mean P@{k}': mean_p_at_k,
        f'Mean R@{k}': mean_r_at_k,
        'MRR': mrr
    }

### 3.1 Default Tokenizer (search_engine)

In [36]:
default_results, default_metrics = evaluate_search_engine(search_engine, eval_df, k=5)

print('=== Default Tokenizer — Per-Query Results ===')
display(default_results[['query_id', 'query', 'P@5', 'R@5', 'RR']])
print(f"\nMean P@5:  {default_metrics['Mean P@5']:.4f}")
print(f"Mean R@5:  {default_metrics['Mean R@5']:.4f}")
print(f"MRR:       {default_metrics['MRR']:.4f}")

=== Default Tokenizer — Per-Query Results ===


,query_id,query,P@5,R@5,RR
0,1,education university college students academic...,0.0,0.0,0.000000
1,2,car vehicle driving automotive auto insurance,0.2,0.2,0.333333
2,3,real estate property housing apartment bedroom...,0.0,0.0,0.000000
3,4,health medical doctor patient disease hospital,0.2,0.2,0.200000
4,5,software computer programming developer algori...,0.0,0.0,0.000000
5,6,recipe cooking delicious ingredient restaurant...,0.2,0.2,0.250000
6,7,travel flight hotel tourist destination vacation,0.0,0.0,0.000000
7,8,sports fitness workout football basketball ath...,0.0,0.0,0.000000
8,9,fashion clothing wardrobe outfit boutique apparel,0.0,0.0,0.000000
9,10,investment finance stock market portfolio banking,0.0,0.0,0.000000



Mean P@5:  0.0600
Mean R@5:  0.0600
MRR:       0.0783


### 3.2 BERT Subword Tokenizer (search_engine_bert)

In [ ]:
bert_results, bert_metrics = evaluate_search_engine(search_engine_bert, eval_df, k=5)
print('=== BERT Subword Tokenizer — Per-Query Results ===')
display(bert_results[['query_id', 'query', 'P@5', 'R@5', 'RR']])
print(f"\nMean P@5:  {bert_metrics['Mean P@5']:.4f}")
print(f"Mean R@5:  {bert_metrics['Mean R@5']:.4f}")
print(f"MRR:       {bert_metrics['MRR']:.4f}")

=== BERT Subword Tokenizer — Per-Query Results ===


,query_id,query,P@5,R@5,RR
0,1,education university college students academic...,0.0,0.0,0.00
1,2,car vehicle driving automotive auto insurance,0.2,0.2,0.25
2,3,real estate property housing apartment bedroom...,0.0,0.0,0.00
3,4,health medical doctor patient disease hospital,0.2,0.2,0.20
4,5,software computer programming developer algori...,0.0,0.0,0.00
5,6,recipe cooking delicious ingredient restaurant...,0.2,0.2,0.20
6,7,travel flight hotel tourist destination vacation,0.0,0.0,0.00
7,8,sports fitness workout football basketball ath...,0.0,0.0,0.00
8,9,fashion clothing wardrobe outfit boutique apparel,0.0,0.0,0.00
9,10,investment finance stock market portfolio banking,0.0,0.0,0.00



Mean P@5:  0.0600
Mean R@5:  0.0600
MRR:       0.0650


## 4. Comparison

In [39]:
comparison = pd.DataFrame({
    'Default Tokenizer': default_metrics,
    'BERT Subword Tokenizer': bert_metrics
}).T

print('=== Search Engine Comparison ===')
display(comparison)

comparison.to_csv("results.csv")

=== Search Engine Comparison ===


,Mean P@5,Mean R@5,MRR
Default Tokenizer,0.06,0.06,0.078333
BERT Subword Tokenizer,0.06,0.06,0.065000
